# Донавчання детектора зброї на Kaggle GPU — Sentinel

Сесія 03, крок 8. **Ноутбук підготовлено, але не запускався:** Kaggle-токена на машині немає (30.09.2026).

- **Старт** — ваги WeaponShield YOLOv8s (класи `Pistol` / `Rifle` / `Knife` лишаються, тож голова моделі не
  переініціалізується); у Sentinel класи зводяться до `gun` / `knife` через `weapons.class_map`.
- **Дані завантажуються тут, з публічних джерел, а не з машини розробника:**
  - Севілья, «Mock attack dataset» (`weapons_images_2fps.zip`), лише **Cam1 + Cam5**, поділ train / val **за
    сегментами** з `weapon_finetune.json`;
  - синтетичний Unity-набір тих самих авторів (`Unity/split-1000.zip`, U1) — лише train;
  - «важкі негативи» — кадри NonFight **нашого train** RWF (дзеркало `DanJoshua/RWF-2000`), де поточний детектор
    дає хибну зброю; вікна перелічено в `weapon_finetune.json`, вирізки навколо хибних рамок — фон без міток.
- **Формат:** повні кадри (навчання на 640) + вирізки навколо людей 320–416 px (як режим `crops`).
- **Ніколи:** Cam7 (відкладений тест) і тест RWF (оригінальний `val`) — перевіряється в коді.
- **Ліцензії:** Севілья й Unity — CC BY-NC 4.0; RWF-2000 — лише для досліджень. Результати не публікувати.
- **Вхід:** невеликий приватний Kaggle-набір `sentinel-weapon-finetune` з одним файлом `weapon_finetune.json`
  (результат `uv run sentinel eval weapon-finetune`: лише списки, без зображень).
- **Вихід:** `/kaggle/working/weaponshield_ft_best.pt` і `metrics.json` — заберемо в сесії 04.

In [ ]:
!pip install -q "ultralytics==8.4.164" "huggingface_hub>=0.34"
import torch

print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
import hashlib
import json
import random
import shutil
import tarfile
import urllib.request
import xml.etree.ElementTree as ET
import zipfile
from pathlib import Path

import cv2
import numpy as np
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

SEED = 2026
random.seed(SEED)
WORK = Path("/kaggle/working")
RAW = WORK / "raw"
DATA = WORK / "weapon_ft"
META = json.loads(Path("/kaggle/input/sentinel-weapon-finetune/weapon_finetune.json").read_text())

CLASSES = ["Pistol", "Rifle", "Knife"]  # the classes of the WeaponShield checkpoint, kept as they are
NAME_TO_CLASS = {"handgun": 0, "pistol": 0, "rifle": 1, "short_rifle": 1, "knife": 2}
HF_SEVILLA = "jsalazar/US-Real-time-gun-detection-in-CCTV-An-open-problem-dataset"
HF_RWF = "DanJoshua/RWF-2000"
SHA256 = {  # LFS hashes from the Hugging Face repos and our local copies of the weights
    "weapons_images_2fps.zip": "558f5baa109abe8fba57b61fa177c75358001f5499f5e78178f4bf7dc9f55e8c",
    "Unity/split-1000.zip": "6f901d1d75990f8b2c21a2617113e66d520ab07911e4b63bac21a35f281b8fbb",
    "RWF-2000.tar.gz": "227c01e017ee3df8753388f81bf0ab65705f5009f3f0e29f7db18bacd81a2d50",
    "weaponshield_best.pt": "0a96416a2f22cec4a3cc0232747a4ed98edb6fa6b4deb8978dd8c7ba7d944934",
    "yolo11n.pt": "0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1",
}
CROP_PAD = 0.25  # as `weapons.crop_pad` of Sentinel's crops mode
for sub in ("images", "labels"):
    for split in ("train", "val"):
        (DATA / sub / split).mkdir(parents=True, exist_ok=True)
RAW.mkdir(parents=True, exist_ok=True)


def check(path: Path, key: str) -> Path:
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            digest.update(block)
    assert digest.hexdigest() == SHA256[key], f"{path}: SHA-256 differs from the expected one"
    return path


def fetch(url: str, name: str) -> Path:
    path = RAW / name
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return check(path, name)

In [ ]:
weaponshield = fetch(
    "https://raw.githubusercontent.com/Halok600/WeaponShield-AI/"
    "7a0b08ed8961a70fe8ceeeedb1b341584eff66f8/model/best.pt",
    "weaponshield_best.pt",
)
person_model = fetch(
    "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11n.pt", "yolo11n.pt"
)

In [ ]:
def voc_boxes(xml_text: str) -> list[tuple[int, float, float, float, float]]:
    """(class, x1, y1, x2, y2) of the weapon objects of a VOC annotation."""
    boxes = []
    for obj in ET.fromstring(xml_text).iter("object"):
        cls = NAME_TO_CLASS.get(obj.findtext("name", "").strip().lower())
        if cls is None:
            continue
        b = obj.find("bndbox")
        boxes.append((cls, *(float(b.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax"))))
    return boxes


def yolo_lines(boxes, w: int, h: int) -> list[str]:
    return [
        f"{c} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}"
        for c, x1, y1, x2, y2 in boxes
    ]


def save(image, lines: list[str], split: str, name: str) -> None:
    cv2.imwrite(str(DATA / "images" / split / f"{name}.jpg"), image)
    (DATA / "labels" / split / f"{name}.txt").write_text("\n".join(lines))


def person_squares(detector, image) -> list[tuple[int, int, int, int]]:
    """Padded squares around people, as Sentinel's crops mode cuts them."""
    h, w = image.shape[:2]
    result = detector.predict(image, imgsz=480, conf=0.25, classes=[0], verbose=False)[0]
    squares = []
    for x1, y1, x2, y2 in result.boxes.xyxy.cpu().numpy():
        side = min(int(max(x2 - x1, y2 - y1) * (1 + CROP_PAD)), w, h)
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        sx = min(max(int(cx - side / 2), 0), w - side)
        sy = min(max(int(cy - side / 2), 0), h - side)
        squares.append((sx, sy, sx + side, sy + side))
    return squares


def add_frame_and_crops(detector, image, boxes, split: str, name: str) -> None:
    """The full frame, plus 320-416 px crops around people with the weapon boxes moved into them."""
    h, w = image.shape[:2]
    save(image, yolo_lines(boxes, w, h), split, name)
    for i, (sx, sy, ex, ey) in enumerate(person_squares(detector, image)):
        inside = []
        for c, x1, y1, x2, y2 in boxes:
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            if sx <= cx <= ex and sy <= cy <= ey:
                inside.append((c, max(x1, sx) - sx, max(y1, sy) - sy, min(x2, ex) - sx, min(y2, ey) - sy))
        size = random.randint(320, 416)
        scale = size / (ex - sx)
        crop = cv2.resize(image[sy:ey, sx:ex], (size, size))
        scaled = [(c, x1 * scale, y1 * scale, x2 * scale, y2 * scale) for c, x1, y1, x2, y2 in inside]
        save(crop, yolo_lines(scaled, size, size), split, f"{name}_crop{i}")

In [ ]:
people = YOLO(str(person_model))

# Sevilla mock attacks: Cam1 + Cam5 only, split by whole segments
split_of = {s: "train" for s in META["sevilla_segments"]["train"]} | {
    s: "val" for s in META["sevilla_segments"]["val"]
}
zip_path = check(
    Path(hf_hub_download(HF_SEVILLA, "weapons_images_2fps.zip", repo_type="dataset")),
    "weapons_images_2fps.zip",
)
with zipfile.ZipFile(zip_path) as z:
    names = z.namelist()
    for name in [n for n in names if n.endswith(".jpg")]:
        stem = Path(name).stem
        assert not stem.startswith("Cam7"), "Cam7 is the deferred test: never train on it"
        if not stem.startswith(("Cam1", "Cam5")):
            continue
        segment = stem.split("_frame_")[0]
        split = split_of.get(segment)
        if split is None:
            continue
        xml_name = name[:-4] + ".xml"
        boxes = voc_boxes(z.read(xml_name).decode()) if xml_name in names else []
        image = cv2.imdecode(np.frombuffer(z.read(name), np.uint8), cv2.IMREAD_COLOR)
        add_frame_and_crops(people, image, boxes, split, f"sev_{stem}")
print({s: len(list((DATA / "images" / s).glob("*.jpg"))) for s in ("train", "val")})

In [ ]:
# the authors' synthetic Unity CCTV frames (U1), training only
unity = check(
    Path(hf_hub_download(HF_SEVILLA, "Unity/split-1000.zip", repo_type="dataset")), "Unity/split-1000.zip"
)
with zipfile.ZipFile(unity) as z:
    names = set(z.namelist())
    for name in sorted(n for n in names if n.endswith(".jpg")):
        xml_name = name[:-4] + ".xml"
        if xml_name not in names:
            continue
        image = cv2.imdecode(np.frombuffer(z.read(name), np.uint8), cv2.IMREAD_COLOR)
        boxes = voc_boxes(z.read(xml_name).decode())
        add_frame_and_crops(people, image, boxes, "train", f"unity_{Path(name).stem}")
print(len(list((DATA / "images" / "train").glob("unity_*.jpg"))), "Unity images")

In [ ]:
# hard negatives: our RWF train NonFight windows where the current detector fires; background crops, no labels
windows = META["hard_negatives"]["windows"]
min_conf = META["hard_negatives"]["min_conf"]
assert all(w["clip"].startswith("train/NonFight/") for w in windows), "only our RWF train NonFight clips"
wanted = {w["clip"] for w in windows}
rwf = check(Path(hf_hub_download(HF_RWF, "RWF-2000.tar.gz", repo_type="dataset")), "RWF-2000.tar.gz")
clips_dir = RAW / "rwf"
with tarfile.open(rwf) as tar:
    for member in tar:
        rel = member.name.split("RWF-2000/", 1)[-1]
        if rel in wanted:
            member.name = rel
            tar.extract(member, clips_dir)
detector = YOLO(str(weaponshield))
negatives = 0
for w in windows:
    cap = cv2.VideoCapture(str(clips_dir / w["clip"]))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(w["t_start"] * fps))
    for k in range(int((w["t_end"] - w["t_start"]) * 10)):  # the 10 frames/s analysis grid
        cap.set(cv2.CAP_PROP_POS_FRAMES, int((w["t_start"] + k / 10) * fps))
        ok, image = cap.read()
        if not ok:
            break
        h, wd = image.shape[:2]
        found = detector.predict(image, imgsz=480, conf=min_conf, verbose=False)[0].boxes.xyxy.cpu().numpy()
        for x1, y1, x2, y2 in found:
            side = min(max(int(max(x2 - x1, y2 - y1) * 4), 64), wd, h)
            sx = min(max(int((x1 + x2) / 2 - side / 2), 0), wd - side)
            sy = min(max(int((y1 + y2) / 2 - side / 2), 0), h - side)
            size = random.randint(320, 416)
            crop = cv2.resize(image[sy : sy + side, sx : sx + side], (size, size))
            save(crop, [], "train", f"neg_{negatives}")
            negatives += 1
    cap.release()
print(negatives, "hard-negative crops")

In [ ]:
(DATA / "data.yaml").write_text(
    f"path: {DATA}\ntrain: images/train\nval: images/val\nnames:\n"
    + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASSES))
)
model = YOLO(str(weaponshield))
model.train(
    data=str(DATA / "data.yaml"),
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    seed=SEED,
    deterministic=True,
    cos_lr=True,
    close_mosaic=10,
    workers=2,
    project=str(WORK / "runs"),
    name="weaponshield_ft",
    exist_ok=True,
)

In [ ]:
best = WORK / "runs" / "weaponshield_ft" / "weights" / "best.pt"
metrics = YOLO(str(best)).val(data=str(DATA / "data.yaml"), split="val", imgsz=640)
summary = {
    "map50": float(metrics.box.map50),
    "map50_95": float(metrics.box.map),
    "per_class_map50": dict(zip(CLASSES, map(float, metrics.box.ap50), strict=False)),
    "val": "Sevilla Cam1 + Cam5 val segments (full frames + person crops)",
    "seed": SEED,
}
(WORK / "metrics.json").write_text(json.dumps(summary, indent=1))
shutil.copy(best, WORK / "weaponshield_ft_best.pt")
print(summary)

## Далі (сесія 04)

- Забрати `weaponshield_ft_best.pt` і `metrics.json`; записати джерело й хеш у `data/SOURCES.md`.
- Перевірити в Sentinel на **Севільї Cam1 + Cam5** (`sentinel eval weapons`, правило «зброя в руці» на кеші) і на
  хибних тривогах NonFight нашого train RWF — тими самими командами, що й для готових ваг.
- Cam7 і тест RWF — лише в експерименті, після вибору ваг.